<a href="https://colab.research.google.com/github/camghal18/Activity-4.2/blob/main/act4_2_trees_and_forests_in_sklearn_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install composable

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import composable.records as rec

In [ ]:
# Standard imports
import polars as pl
import polars.selectors as cs
import seaborn as sns
import numpy as np

from sklearn.preprocessing import StandardScaler

# Pipeline
from sklearn.pipeline import Pipeline


# Preprocessing stuff
from sklearn.preprocessing import LabelEncoder

# Model selection stuff
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV

# Classic classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier


# Metrics to use on the test set
# metric(y_test, y_predict)
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score




# Trees and Forests in `sklearn`

In this notebook, we will cover the basics of classification, including:

1. The `DecisionTreeClassifier` and `RandomForestClassifier`.
2. Exploring tuning parameters for each model.
3. Performing a grid search.
4. Additional features of Random Forests.
5. Multiclass problems.

## Classification Example: Kyphosis Data Set

**Problem Statement:**
Given a DataSet of 81 patients who have undergone a spinal surgery for a deformation and the data if the condition recurred, Build a claddification Model to predict whether a patient being admitted for the surgery has chance for recurrence. This model will help the surgeons to plan appropriate level of treatment to prevent recurrence.

**Data Set Description :**
The kyphosis data frame has 81 rows and 4 columns. representing data on children who have had corrective spinal surgery

This data frame contains the following columns/Features:

1. *Kyphosis*: a factor with levels absent present indicating if a kyphosis (a type of deformation) was present after the operation.
2. *Age*: in months
3. *Number*: the number of vertebrae involved
4. *Start*: the number of the first (topmost) vertebra operated on.

**Research Question:** Can we predict whether if kyphosis will be present or absent after the surgery based on the age of the patient, number of vertebrae involved and the first vertebra operated on?

In [ ]:
(kyphosis :=
 pl.read_csv('sample_data/kyphosis.csv')
   .drop('rownames')
)

In [ ]:
(X_kyphosis :=
 kyphosis
 .drop('Kyphosis')
 .to_pandas()
)

In [ ]:
(y_kyphosis :=
 kyphosis
 .get_column('Kyphosis')
 .to_numpy()
 .ravel()
)

## Topic 1 - Tree and Forest models in `sklearn`

`sklearn` comes with both a tree and forest based classifier, which can be found in the `tree` and `ensemble` submodules, respectively.

In [ ]:
(tree := DecisionTreeClassifier(class_weight='balanced')
)

In [ ]:
(forest := RandomForestClassifier(class_weight='balanced')
)

## Tuning parameters for trees and forests

First, let's inspect the tuning parameters for each model.

#### Exploring trees

In [ ]:
?tree

In [ ]:
tree.get_params()

In [ ]:
(tree_grid :=
 {'min_samples_leaf': [1,2,3],
  'min_samples_split': [2,3,4],
  'max_depth': [3,4,5],
 }
)

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
grid_search_tree = GridSearchCV(tree, tree_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_tree.fit(X_kyphosis, y_kyphosis)


In [ ]:
?cross_validate

In [ ]:
my_scores = ['accuracy', 'balanced_accuracy', 'roc_auc']

(tree_scores :=
 cross_validate(grid_search_tree, X_kyphosis, y_kyphosis,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)

#### Exploring forests

In [ ]:
?forest

In [ ]:
forest.get_params()

In [ ]:
(forest_grid :=
{'max_depth': [2,3,4,5],
 'min_samples_leaf': [1,2,3],
 'min_samples_split': [2,3,4],
 'n_estimators': [10, 100, 500],
 }
)

In [ ]:
grid_search_forest = GridSearchCV(forest, forest_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_forest.fit(X_kyphosis, y_kyphosis)

In [ ]:
(forest_scores :=
 cross_validate(grid_search_tree, X_kyphosis, y_kyphosis,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)


## Topic 3 - Extra random forest features

As you learned from your out of class videos, the fact that random forests uses bagging allows for extra functionality, including

1. A measure of variable importance, and
2. An out of the box objective measurement of model fit.

#### Variable importance

In [ ]:
grid_search_forest.best_estimator_.feature_importances_ # Ugh! so unreadable!

In [ ]:
(feat_imp :=
 pl.DataFrame({'columns':grid_search_forest.feature_names_in_,
               'importance':grid_search_forest.best_estimator_.feature_importances_},
             )
   .sort('importance', descending=True)
)


#### Out of bag error estimate

**Notes.**

1. You need to set `oob_score=True` when creating the forest to get this measure.
2. Computing the `oob_score` has some overhead, so it is best leave this off when performing a grid search.  Instead, refit the winning model with this flag after.

In [ ]:
grid_search_forest.best_params_

In [ ]:
(winning_forest_w_oob_error :=
 RandomForestClassifier(oob_score=True,

                       **grid_search_forest.best_params_,
                      )

)

In [ ]:
winning_forest_w_oob_error.fit(X_kyphosis, y_kyphosis)

winning_forest_w_oob_error.oob_score_

## Topic 4 - Multiclass classification

In [ ]:
(olive_oil :=
 pl.read_csv("sample_data/OliveOils.csv")
).head(2)

In [ ]:
(X_oil :=
 olive_oil
 .drop('Area.name')
 .to_pandas()
).head(2)

In [ ]:
(y_oil :=
 olive_oil
 .select('Area.name')
 .to_numpy()
 .ravel()
)[:3]

In [ ]:
X_train_oil, X_test_oil, y_train_oil, y_test_oil = train_test_split(X_oil, y_oil, test_size=0.3, random_state=42, stratify=y_oil)

In [ ]:
grid_search_tree_MC = GridSearchCV(tree, tree_grid, cv = folds, scoring='balanced_accuracy', n_jobs=-1, verbose=0)

grid_search_tree_MC.fit(X_train_oil, y_train_oil)
grid_search_tree_MC.score(X_test_oil, y_test_oil)

In [ ]:
grid_search_forest_MC = GridSearchCV(forest, forest_grid, cv = folds, scoring='balanced_accuracy', n_jobs=-1, verbose=0)

grid_search_forest_MC.fit(X_train_oil, y_train_oil)
grid_search_forest_MC.score(X_test_oil, y_test_oil)

## <font color='red'> Exercise 1 </font>

Do some sleuthing to determine the impact of various tuning parameters for both trees and forests on the model flexibility and overfitting.

<font color='orange'>
A summary of your findings here.
</font>

In [ ]:
?tree

In [ ]:
tree.get_params()

In [ ]:
?forest

 #### The impact of certain parameters for the tree classification model such as criterion, splitter and max_depth params. The ccp_alpha parameter is able to chop off branches that don't add predictive power uses the measure to the quality of a split, the min_samples_leaf is used to choose the split at each node, and the max_depth (which is the most important) deals with overfitting.

#### The impact of certain parameters for the forest classification model such as n_estimator determines  the number of trees, the max features how many limits each tree can look for a split, and the bootstrap as the tree uses sub-samples which averages out the mean between trees.  

## <font color="red"> Exercise 2 </font>

1. Use a combined grid search to compare the performance of trees and forests on the Pima Indian diabetes data set.  
2. Validate the winning model by computing CV scores on the test set.
3. Use random forests to explore the feature importance.  If a forest wasn't the winning model, redo the grid search to find the best forest.
4. Comment on your findings.

In [ ]:
(
    diabetes :=
 pl.read_csv('sample_data/diabetes_raw.csv')
)

In [ ]:
(X_diabetes :=
    diabetes
    .drop('Diabetes')
    .to_pandas()
 )

In [ ]:
( y_diabetes :=
       diabetes
       .select('Diabetes')
       .to_pandas()
)

In [ ]:
X_train_diabetes, X_test_diabetes, y_train_diabetes, y_test_diabetes = train_test_split(X_diabetes,y_diabetes, test_size=0.3, random_state=42)

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
(tree := DecisionTreeClassifier(class_weight = 'balanced')
)

In [ ]:
grid_search_forest = GridSearchCV(forest, forest_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

grid_search_forest.fit(X_diabetes, y_diabetes)

In [ ]:
(tree_grid :=
  { 'classifier': [DecisionTreeClassifier()], # Kept in a list to avoid that previous error!
    'classifier__min_samples_leaf': [1, 2, 3],
    'classifier__min_samples_split': [2, 3, 4],
    'classifier__max_depth': [3, 4, 5]
 }
)

In [ ]:
(forest_grid :=
{
    'classifier': [RandomForestClassifier()], # Added the 's' in classifier
    'classifier__max_depth': [2, 3, 4, 5],
    'classifier__min_samples_leaf': [1, 2, 3],
    'classifier__min_samples_split': [2, 3, 4],
    'classifier__n_estimators': [10, 100, 500]
}
)

In [ ]:
(generic_cls :=
 Pipeline(steps = [
     ('scaler', StandardScaler()),
     ('classifier', None)  # <-- classifier goes here
 ])

)

In [ ]:
(combined_grid :=

 [forest_grid, tree_grid]


)

In [ ]:
(combined_grid_search :=
 GridSearchCV(generic_cls,combined_grid, cv = folds, scoring='balanced_accuracy', verbose=1 )

 )

In [ ]:
combined_grid_search = GridSearchCV(generic_cls, combined_grid, cv = folds, scoring='roc_auc', n_jobs=-1, verbose=1)

combined_grid_search.fit(X_train_diabetes, y_train_diabetes)

In [ ]:
grid_search_tree.best_params_

In [ ]:
(combined_grid :=

 [forest_grid, tree_grid]


)

In [ ]:
my_scores = ['accuracy', 'balanced_accuracy', 'roc_auc']



(diabetes_scores :=
 cross_validate(combined_grid_search, X_test_diabetes, y_test_diabetes,
                cv = folds,
                scoring=my_scores,
                verbose=0,
                n_jobs=-1,
                )
 >> rec.subset([f'test_{m}' for m in my_scores])
 >> rec.map(np.mean)
)



<font color="orange">
Your summary here.
</font>

### The best classifier out of the two options (random forest and decision tree) was the random forest classifier which was able to predict the classes with max depth of 5, classifier min sample leaf of 2, classifier min sample split of 2.  The cross validation map for our scores went as follows:
### test_accuracy: 0.71
### test_balanced_accuracy: 0.64
### test_roc_auc : NaN

## <font color="red"> Exercise 3 </font>

Now we build on the final exercise of the previous activity, where we performed a combined grid search over all of the classic classification methods.  Redo this, but this time add trees and forests of each type (alone, OvR, and OVO).

1. Create a combined grid on all 15 classifiers (5 classic classifiers + 5*OvR + 5*OvO).  
2. For `kNN`, add a number of `metric`s and `weights` to the grid.
3. Add 6 tree based classifiers to the grid search (tree + forest as well as OvR and OvO for each).
4. Perform the grid search over all the models and determing the winning model.
5. Evaluate the performance of the winning model using CV and write a summary interpreting each metric.

In [ ]:
(knn_alone :=
  {'classifier': [KNeighborsClassifier()],
   'classifier__n_neighbors': list(range(2,12,2)),
    'classifier__metric': ['l1','l2'],
    'classifier__weights': ['uniform','distance']
  }
 )

In [ ]:
knn_wrapped = {
    'classifier': [
        OneVsRestClassifier(KNeighborsClassifier()),
        OneVsOneClassifier(KNeighborsClassifier())
    ],
    'classifier__estimator__n_neighbors': list(range(2,12,2)),
    'classifier__estimator__metric': ['l1','l2'],
    'classifier__estimator__weights': ['uniform','distance']
}

In [ ]:
#(knn_wrapped :=
#    {'classifier': [OneVsOneClassifier(KNeighborsClassifier()),OneVsOneClassifier(KNeighborsClassifier())],
#     'classifier__estimator__n_neighbors': list(range(2,12,2)),
#     'classifier__estimator__metric': ['l1','l2'],
#     'classifier__estimator__weights': ['uniform','distance']
#    }
#)

In [ ]:
(knn_grid :=
[knn_alone, knn_wrapped]
)

In [ ]:
# Your code here (add cells as needed.)


(nontuning_grid :=  {'classifier' :
[LogisticRegression(max_iter=10000),
 OneVsRestClassifier(LogisticRegression(max_iter=10000)),
 OneVsOneClassifier(LogisticRegression(max_iter=10000)),
 GaussianNB(),
 OneVsRestClassifier(GaussianNB()),
 OneVsOneClassifier(GaussianNB()),
 LinearDiscriminantAnalysis(),
 OneVsRestClassifier(LinearDiscriminantAnalysis()),
 OneVsOneClassifier(LinearDiscriminantAnalysis()),
 QuadraticDiscriminantAnalysis(),
 OneVsRestClassifier(QuadraticDiscriminantAnalysis()),
 OneVsOneClassifier(QuadraticDiscriminantAnalysis())

]
}
)

In [ ]:
DecisionTreeClassifier().get_params()

In [ ]:
(
 tree_alone :=
 { 'classifier' : [DecisionTreeClassifier()],
    'classifier__max_depth': [3, 5, 10, 15, 20],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__min_samples_split': [2, 5, 10]
 }
)

In [ ]:
(
 tree_wrapped :=

 { 'classifier' : [OneVsRestClassifier(DecisionTreeClassifier()),
                   OneVsOneClassifier(DecisionTreeClassifier())],
    'classifier__estimator__max_depth': [3, 5, 10, 15, 20],
    'classifier__estimator__min_samples_leaf': [1, 2, 4],
    'classifier__estimator__min_samples_split': [2, 5, 10]
 }

)

In [ ]:
(tree_grid :=

 [tree_alone,tree_wrapped]


)

In [ ]:
(
 forest_alone := {
    'classifier': [RandomForestClassifier()],
    'classifier__n_estimators': [50, 100, 200, 500],
    'classifier__max_depth': [5, 10, 15, 20],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__min_samples_split': [2, 5, 10]
}

)

In [ ]:
(
 forest_wrapped := {
    'classifier': [OneVsRestClassifier(RandomForestClassifier()),
                    OneVsOneClassifier(RandomForestClassifier())],
    'classifier__estimator__n_estimators': [50, 100, 200, 500],
    'classifier__estimator__max_depth': [5, 10, 15, 20],
    'classifier__estimator__min_samples_leaf': [1, 2, 4],
    'classifier__estimator__min_samples_split': [2, 5, 10]
}

)

In [ ]:
(forest_grid :=

    [forest_alone,forest_wrapped]

 )

In [ ]:
#(combined_grid :=

#[forest_grid,tree_grid,knn_grid,nontuning_grid]
#)

In [ ]:
(combined_grid :=

forest_grid + tree_grid + knn_grid + [nontuning_grid]
#[forest_grid,tree_grid,knn_grid,nontuning_grid]



)

In [ ]:
(combined_grid_search :=
GridSearchCV(generic_cls, combined_grid,cv=folds, scoring='balanced_accuracy', verbose=1)
)

In [ ]:
(combined_grid_search.fit(X_train_diabetes, y_train_diabetes))

In [ ]:
print(f"Total models searched: {len(combined_grid_search.cv_results_['params'])}")
import pandas as pd
df = pd.DataFrame(combined_grid_search.cv_results_)
# This sorts them so you can see if QDA is at #2 or #20
print(df[['param_classifier', 'mean_test_score']].sort_values('mean_test_score', ascending=False))

In [ ]:
metrics = ['accuracy',
           'balanced_accuracy',
           'f1_micro'
]


(all_combined_test_scores :=
 cross_validate(combined_grid_search, X_test_diabetes,y_test_diabetes,
                cv = folds,
                scoring = metrics,
                verbose =1,
                n_jobs = -1
 )
  >>rec.subset([f'test_{m}' for m in metrics])
  >>rec.map(np.mean)
 )


### Overall, the best classifier of the 89 models was the Random Forest Classifier with a test score of 0.74, the parameters of the classifier including a n_estimator of 50, min samples split of 5 and a max depth of 10. The metrics for the cross validation of the combined grid search went as follows:


### 'test_accuracy':  0.75   
### 'test_balanced_accuracy' : 0.71
###  'test_f1_micro' : 0.75